# Attribution and reuse

This notebook documents the data workflow used to collect, structure and analyse High Court judgments from the **[Eswatini Legal Information Institute (EswatiniLII)](https://eswatinilii.org/)** as part of the **[Without Justice](https://veza.news/article/2024/10/24/without-justice-how-eswatinis-system-is-failing-victims-of-gender-based-violence/)** investigation led by the **[Center for Collaborative Investigative Journalism (CCIJ)](https://ccij.io/)**.

The methodology, including web scraping, document processing and text analysis, was developed and implemented by **Sotiris Sideris** to support the identification and review of **gender-based violence (GBV)** cases within court records.

Parts of the workflow rely on publicly available court records and document analysis tools as well as collaborative reporting processes within investigative newsrooms.

The code and methodology are shared here for transparency and reproducibility.

Please cite or attribute if reusing this method in reporting or research.

## Building a gender-based violence GBV case dataset from Eswatini High Court judgments

This notebook documents a reproducible workflow used to collect, structure and analyze court judgments from the Eswatini Legal Information Institute (EswatiniLII).

The goal is to build a searchable dataset of cases that can be used to identify and analyze GBV cases, including sexual assault, rape, and domestic violence.

Pipeline overview:

1. Scrape case listings from EswatiniLII
2. Extract detailed case metadata
3. Generate standardized PDF download links
4. Download judgment PDFs
5. Extract text from PDFs (direct extraction + OCR fallback)
6. Merge extracted text with structured case metadata
7. Produce a dataset for GBV analysis

## Environment and dependencies

This notebook uses Python libraries for scraping, PDF parsing, OCR and tabular data processing.

Python packages:
- pandas
- requests
- beautifulsoup4
- selenium
- pypdf
- pytesseract
- pdf2image

System dependencies:
- Tesseract OCR
- Poppler

Some cells also use absolute local paths. If you are running the notebook on another machine, update those path variables before executing the download and PDF-processing stages.


## Expected outputs

Running the notebook produces three main outputs:

- a structured case-metadata table
- a text corpus extracted from downloaded judgment PDFs
- a merged analysis dataset linking case metadata to document text

These outputs can then be used for keyword filtering, tagging or model-assisted classification of GBV-related cases.


### Re-running this notebook

This notebook is written as a reproducible workflow, but a few steps depend on local files and folders. 

The scraping steps themselves can be reproduced without those files.

Before re-running it in a new environment, update the local paths used for:

- downloaded PDF storage
- helper files such as `unique_urls.txt`
- optional Google Pinpoint HTML exports

The core workflow remains the same: scrape metadata, prepare download links, retrieve PDFs, extract text and merge the text corpus back into the case metadata.

## 1) Imports and initial setup

In [ ]:
# Import our working libraries

import os
import re
import time
from pathlib import Path
from urllib.parse import urljoin
import pandas as pd
import requests
from bs4 import BeautifulSoup
from selenium import webdriver

from pypdf import PdfReader
import pytesseract
from pdf2image import convert_from_path

In [ ]:
# Display settings for easier inspection while developing

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

In [ ]:
# Open the EswatiniLII High Court listing page
# Launch a browser instance to access dynamically rendered content

driver = webdriver.Chrome()
driver.get('https://eswatinilii.org/judgments/SZHC/')

## 2) Scrape case listing pages

The EswatiniLII judgments page lists cases across multiple pages.  

This step collects the basic case records available on the listing pages, including:

- case title
- case URL
- judgment date

These links are later used to scrape detailed metadata from each case page.

In [ ]:
BASE_URL = "https://eswatinilii.org"

entries_all = []

start_page = 1
end_page_exclusive = 5

for page in range(start_page, end_page_exclusive):
    url = f"{BASE_URL}/judgments/SZHC/?page={page}"
    driver.get(url)
    time.sleep(3)

    doc = BeautifulSoup(driver.page_source, "html.parser")

    # Target the correct table
    table = doc.select_one("table.doc-table")
    if not table:
        print("No doc-table found on page:", page, url)
        break

    # Each judgment row has data-document-id; group header rows do not
    rows = table.select("tbody tr[data-document-id]")

    if not rows:
        print("No document rows found on page:", page, url)
        break

    for r in rows:
        all_entries_dict = {}

        a = r.select_one("td.cell-title a[href]")
        d = r.select_one("td.cell-date")

        if a:
            title = a.get_text(" ", strip=True)
            href = a.get("href")
            all_entries_dict["Case_Title"] = title
            all_entries_dict["Case_Link"] = urljoin(BASE_URL, href) if href else None

        if d:
            all_entries_dict["Case_Date"] = d.get_text(" ", strip=True) or None

        # keep only rows that actually have a title/link
        if all_entries_dict.get("Case_Title"):
            entries_all.append(all_entries_dict)
            print(all_entries_dict)
            print("-----")

    # polite pacing
    time.sleep(1)

print("Total scraped:", len(entries_all))


In [ ]:
# Convert the collected list of case records into a Pandas DataFrame
# Remove duplicate cases that may appear across multiple pagination pages. The case URL serves as a unique identifier for each judgment.
# Display the resulting dataset

df = pd.DataFrame(entries_all)
df = df.drop_duplicates(subset=['Case_Link'], keep='first')
df

In [ ]:
# Save raw case listing data (Stage 1 dataset)

df.to_csv("01_case_links.csv", index=False)

## 3) Scrape detailed case metadata

Once the case page URLs are known, this block visits each case page and extracts the detailed metadata fields displayed in the judgment container, including:

- court name
- citation
- judges
- hearing dates
- summary

This step visits each case page and extracts those fields.

In [ ]:
# Visit each case page, extract detailed metadata fields and append them to a list

entries_all2 = []

for item in df['Case_Link'][0:10]:  
    if len(item) > 1:
        time.sleep(2)
        driver.get(str(item))
        time.sleep(0.5)
        doc = BeautifulSoup(driver.page_source, 'html.parser')
        each_entry = doc.find('div', class_='container')
        
        for entry in each_entry:
    
            try:
                all_entries_dict2 ={}
                
                all_entries_dict2['Case_Link'] = item
                
                # Jurisdiction
                jurisdiction_dt = doc.find(lambda tag: tag.name == "dt" and "Jurisdiction" in tag.text)
                jurisdiction_dd = jurisdiction_dt.find_next_sibling("dd") if jurisdiction_dt else None
                jurisdiction = jurisdiction_dd.get_text(strip=True) if jurisdiction_dd else "Jurisdiction not found"
                all_entries_dict2['Jurisdiction'] = jurisdiction
                
                # Citation
                citation_dt = doc.find(lambda tag: tag.name == "dt" and "Citation" in tag.text)
                citation_dd = citation_dt.find_next_sibling("dd") if citation_dt else None
                citation = citation_dd.get_text(strip=True) if citation_dd else "Citation not found"
                all_entries_dict2['Citation'] = citation
                
                # Media Neutral Citation
                media_neutral_citation_dt = doc.find(lambda tag: tag.name == "dt" and "Media Neutral Citation" in tag.text)
                media_neutral_citation_dd = media_neutral_citation_dt.find_next_sibling("dd") if media_neutral_citation_dt else None
                media_neutral_citation = media_neutral_citation_dd.get_text(strip=True) if media_neutral_citation_dd else "Media Neutral Citation not found"
                all_entries_dict2['Media_Neutral_Citation'] = media_neutral_citation
                
                # Hearing date
                hearing_date_dt = doc.find(lambda tag: tag.name == "dt" and "Hearing date" in tag.text)
                hearing_date_dd = hearing_date_dt.find_next_sibling("dd") if hearing_date_dt else None
                hearing_date = hearing_date_dd.get_text(strip=True) if hearing_date_dd else "Hearing date not found"
                all_entries_dict2['Hearing_Date'] = hearing_date
                
                # Court
                court_dt = doc.find(lambda tag: tag.name == "dt" and "Court" in tag.text)
                court_dd = court_dt.find_next_sibling("dd") if court_dt else None
                court = court_dd.get_text(strip=True) if court_dd else "Court not found"
                all_entries_dict2['Court'] = court
                
                # Case number
                case_number_dt = doc.find(lambda tag: tag.name == "dt" and "Case number" in tag.text)
                case_number_dd = case_number_dt.find_next_sibling("dd") if case_number_dt else None
                case_number = case_number_dd.get_text(strip=True) if case_number_dd else "Case number not found"
                all_entries_dict2['Case_Number'] = case_number
                
                # Judges
                judges_dt = doc.find(lambda tag: tag.name == "dt" and "Judges" in tag.text)
                judges_dd = judges_dt.find_next_sibling("dd") if judges_dt else None
                judges = judges_dd.get_text(strip=True) if judges_dd else "Judges not found"
                all_entries_dict2['Judges'] = judges
                
                # Judgment date
                judgment_date_dt = doc.find(lambda tag: tag.name == "dt" and "Judgment date" in tag.text)
                judgment_date_dd = judgment_date_dt.find_next_sibling("dd") if judgment_date_dt else None
                judgment_date = judgment_date_dd.get_text(strip=True) if judgment_date_dd else "Judgment date not found"
                all_entries_dict2['Judgment_Date'] = judgment_date
                
                # Language
                language_dt = doc.find(lambda tag: tag.name == "dt" and "Language" in tag.text)
                language_dd = language_dt.find_next_sibling("dd") if language_dt else None
                language = language_dd.get_text(strip=True) if language_dd else "Language not found"
                all_entries_dict2['Language'] = language
                
                # Type
                type_dt = doc.find(lambda tag: tag.name == "dt" and "Type" in tag.text)
                type_dd = type_dt.find_next_sibling("dd") if type_dt else None
                type = type_dd.get_text(strip=True) if type_dd else "Type not found"
                all_entries_dict2['Type'] = type
                
                # Download Links
                download_links = doc.find_all('a')
                download_types = ['RTF', 'PDF', 'DOC']

                for link in download_links:
                    link_text = link.text.strip().upper()
                    for dtype in download_types:
                        if dtype in link_text:
                            href = link.get('href', None)
                            if href:
                                # Store each type of download link in the dictionary
                                all_entries_dict2[f'Download_Link_{dtype}'] = href
                    
                # Other Documents Links   
                other_documents = doc.find(lambda tag: tag.name == "dt" and "Other documents" in tag.text)
                if other_documents:
                    next_sibling = other_documents.find_next_sibling("dd")
                    if next_sibling:
                        link = next_sibling.find('a')
                        href = link.get('href') if link else "Other documents link not found"
                        all_entries_dict2['Other_Documents_Link'] = href
                else:
                    all_entries_dict2['Other_Documents_Link'] = "Other documents section not found"

                # Case Summary
                case_summary_dt = doc.find(lambda tag: tag.name == "dt" and "Case summary" in tag.text)
                case_summary_dd = case_summary_dt.find_next_sibling("dd") if case_summary_dt else None
                case_summary = case_summary_dd.get_text(strip=True) if case_summary_dd else "Case summary not found"
                all_entries_dict2['Case_Summary'] = case_summary.strip()

            except:
                pass
        
    entries_all2.append(all_entries_dict2)
    print(all_entries_dict2)
    print('-----')
    time.sleep(0.5)


In [ ]:
# Merge the listing-page data with the detailed case metadata

an_df = pd.DataFrame(entries_all2)
an_df = df.merge(an_df, left_on="Case_Link", right_on="Case_Link")
an_df

In [ ]:
df.head()
df.columns

In [ ]:
an_df.shape

In [ ]:
# Save the merged case metadata to CSV

an_df.to_csv("02_case_metadata_20.csv", index=False)

## 4) Prepare the scraped case dataset for bulk PDF download

The EswatiniLII site provides a direct PDF version of each judgment at: **{Case_Link}/source.pdf**

This step constructs those download links and generates standardized filenames so the documents can later be matched to the metadata table.

The dataset is then exported to helper files used for bulk downloading the judgment PDFs. 

A Python script retrieves the documents from the source site and stores them locally for further text extraction and analysis.

In [ ]:
df = pd.read_csv("02_case_metadata_20.csv")
df

In [ ]:
df['Citation'] = df['Citation'].str.replace('Copy', '')
df['Media_Neutral_Citation'] = df['Media_Neutral_Citation'].str.replace('Copy', '')

In [ ]:
df['Download_Link'] = df['Case_Link'] + "/source.pdf"

In [ ]:
# Build a direct PDF download link for each case and create a standardized filename
parts = df["Case_Link"].str.extract(
    r"/szhc/(?P<year>\d{4})/(?P<num>\d+)/.*@(?P<date>\d{4}-\d{2}-\d{2})"
)

df["Rename"] = (
    parts["num"] + "_" +
    parts["date"].str.replace("-", "_", regex=False) +
    ".pdf"
)

# Keep the standardized PDF filename in the main dataset for later merging
df["PDF_filename"] = df["Rename"]


In [ ]:
df['Rename'].value_counts()

In [ ]:
# Save the enriched case metadata, including the standardized PDF filename

df.to_csv("03_all_details_20.csv", index=False)

In [ ]:
# Export helper files used for the bulk PDF download step

df.to_csv("unique_urls.txt", columns=['Download_Link', 'Rename'], header=False, index=False)

### Local paths for PDF download

This step uses local output directories for storing helper files and downloaded judgment PDFs.  
If you are re-running the notebook in another environment, update the path variables below to match your local project structure.


In [ ]:
# Base directory for the project (update this path when reproducing the workflow)
BASE_DIR = "/path/to/project_folder"

LIST_FILE = os.path.join(BASE_DIR, "unique_urls.txt")
OUTDIR = os.path.join(BASE_DIR, "eswatini_pdfs")

# Create output folder
os.makedirs(OUTDIR, exist_ok=True)

headers = {
    "User-Agent": "Mozilla/5.0"
}

session = requests.Session()
session.headers.update(headers)

with open(LIST_FILE, "r", encoding="utf-8") as f:
    for line in f:

        line = line.strip()

        if not line:
            continue

        try:
            url, fname = [x.strip() for x in line.split(",", 1)]
        except ValueError:
            print("Skipping malformed line:", line)
            continue

        out_path = os.path.join(OUTDIR, fname)

        print("Downloading:", fname)

        try:
            r = session.get(url, timeout=30)
            r.raise_for_status()

            with open(out_path, "wb") as pdf:
                pdf.write(r.content)

        except Exception as e:
            print("FAILED:", url, "|", e)


print("All done.")


## Optional step - Use Google Pinpoint to identify relevant PDFs before local text extraction

After downloading the full corpus locally, an optional workflow is to upload the files to **Google Pinpoint** and use its OCR and search interface to identify potentially relevant documents before continuing with local processing.

A practical use case for this project was to search for the following keywords:

- `rape`
- `sexual assault`
- `domestic violence`

Once the relevant files were identified in Pinpoint, the filenames of those tagged documents were extracted from the Pinpoint HTML and used to download only the corresponding PDFs locally for further analysis.

### Advantages
- Pinpoint is strong at OCR and can perform well on poor-quality scans or handwritten material.
- The search interface is useful for quickly narrowing a very large corpus to likely relevant files.
- It can reduce the number of files that need to be processed locally.

### Limitations
- Pinpoint does not currently support bulk export of the documents identified through search.
- An additional workaround is required to extract the filenames of the relevant files from the page HTML.
- This makes the step helpful in practice, but less fully reproducible than the local pipeline.

The two optional cells below (before step 5) show how to extract filenames from a Pinpoint HTML export and then download only those relevant PDFs using the same local download logic used earlier in the notebook **(see Local paths for PDF download above)**.


In [ ]:
# Path to saved HTML source from the Pinpoint search results page
PINPOINT_HTML = "/path/to/pinpoint_html_export.txt"

with open(PINPOINT_HTML, "r", encoding="utf-8", errors="ignore") as f:
    html = f.read()

# Pinpoint stores document filenames in the data-tooltip attribute
regex = r'data-tooltip="([^"]+\.pdf)"'
filenames = re.findall(regex, html)

# Convert the extracted filenames to a dataframe for inspection
pinpoint_files = pd.DataFrame(filenames, columns=["PDF_filename"])
pinpoint_files.head()

In [ ]:
# Load the full URL list created earlier in the notebook
all_urls = pd.read_csv(
    os.path.join(BASE_DIR, "unique_urls.csv")
)

# Keep only the files identified as relevant through the Pinpoint search workflow
relevant_urls = all_urls[all_urls["Rename"].isin(pinpoint_files["PDF_filename"])].copy()

# Save the filtered list for reference or reuse
relevant_urls.to_csv(os.path.join(BASE_DIR, "pinpoint_relevant_urls.csv"), index=False)

# Optional: download only the relevant PDFs to a separate local folder
PINPOINT_OUTDIR = os.path.join(BASE_DIR, "eswatini_pdfs_pinpoint")
os.makedirs(PINPOINT_OUTDIR, exist_ok=True)

for _, row in relevant_urls.iterrows():
    url = row["Download_Link"]
    fname = row["Rename"]
    out_path = os.path.join(PINPOINT_OUTDIR, fname)

    print("Downloading:", fname)

    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()

        with open(out_path, "wb") as pdf:
            pdf.write(r.content)

    except Exception as e:
        print("FAILED:", url, "|", e)

print("Pinpoint-selected downloads complete.")

## 5) Extract text from downloaded PDFs

This step processes the downloaded court judgments and converts them into a structured text corpus for analysis.

The workflow follows a two-stage extraction strategy:

1. **Direct text extraction** – the script first attempts to extract text directly from the PDF using `pypdf`. Many court judgments contain embedded text that can be read without additional processing.

2. **OCR fallback** – if the document contains little or no extractable text (for example scanned documents), the script converts each page to an image and performs optical character recognition (OCR) using `Tesseract`.

Each processed file returns:

- the extraction method used (`text_pdf` or `ocr_pdf`)
- the filename
- the number of pages
- the extracted text content

This approach ensures that both digitally generated and scanned documents can be incorporated into the analysis pipeline.

In [ ]:
def process_pdf(filepath, min_text_len=50):
    filename = os.path.basename(filepath)

    # ---- 1. Try normal PDF text extraction ----
    try:
        reader = PdfReader(filepath)
        text = "\n".join(page.extract_text() or "" for page in reader.pages)

        if text and len(text.strip()) >= min_text_len:
            return pd.Series({
                "status": "text_pdf",
                "filename": filename,
                "Pages": len(reader.pages),
                "content": text
            })

    except Exception as e:
        print(f"[PDF ERROR] {filename}: {e}")


    # ---- 2. Fallback: OCR ----
    try:
        images = convert_from_path(filepath, dpi=300)

        texts = []
        for img in images:
            txt = pytesseract.image_to_string(img, lang="eng+ell")
            texts.append(txt)

        full_text = "\n".join(texts)

        return pd.Series({
            "status": "ocr_pdf",
            "filename": filename,
            "Pages": len(images),
            "content": full_text
        })

    except Exception as e:
        print(f"[OCR ERROR] {filename}: {e}")

        return pd.Series({
            "status": "error",
            "filename": filename,
            "Pages": None,
            "content": None
        })


## 6) Process the judgment corpus

The following step loads all downloaded PDF files and applies the extraction function to each document.

Each file is processed sequentially and the results are stored in a dataframe containing:

- filename
- extraction method
- page count
- extracted text

This structured dataset forms the textual corpus used for subsequent analysis of court judgments.

In [ ]:
# Directory containing downloaded judgment PDFs
PDF_DIR = "/path/to/eswatini_pdfs"

# Collect all PDF files in the directory
pdf_files = list(Path(PDF_DIR).glob("*.pdf"))

# Container for processed results
rows = []

# #Process each PDF file
for fp in pdf_files:
    rows.append(process_pdf(str(fp)))

# Convert results to a structured dataframe
df = pd.DataFrame(rows)

### Quick quality checks for extracted text

Before exporting the text corpus, it is useful to inspect how many files were processed through direct extraction, OCR, or error handling.  
This provides a fast quality-control check on the corpus before the merge step.


In [ ]:
# Quick QA checks for the extracted text corpus
df["content_length"] = df["content"].str.len()

print(df["status"].value_counts(dropna=False))
df[["filename", "status", "content_length"]].head()

In [ ]:
# Quick QA check: inspect extracted text length across processed PDFs

df["content_length"] = df["content"].str.len()

In [ ]:
# Show sample of documents that required OCR

ocr_cases = df[df["status"] == "ocr_pdf"]
ocr_cases[["filename", "content"]].head(10)

In [ ]:
# Save the extracted text corpus and metadata to a CSV file for downstream analysis

df.to_csv("04_case_text.csv", index=False)

## 7) Load the structured case and text datasets

This step loads the main case dataset containing metadata for the court judgments.  
Only the relevant columns are selected in order to focus the analysis on key case attributes such as case title, citation, court information and document download links.

The resulting dataframe will later be merged with the extracted PDF text corpus.

In [ ]:
# Load the main case dataset
gbv = pd.read_csv("03_all_details_20.csv")

# Preview the dataset
gbv

In [ ]:
# Load the dataset containing text extracted from the judgment PDFs
pdf_df = pd.read_csv("04_case_text.csv")

# Preview the dataframe
pdf_df

## 8) Create a common filename key and merge datasets

To merge the case metadata with the extracted PDF text, a shared identifier is needed. This step standardizes the PDF filename in both datasets by stripping extra whitespace and converting all characters to lowercase.

Using a common filename key helps ensure that the metadata table and the extracted text corpus can be matched reliably.


In [ ]:
# Standardize the extracted PDF filenames for reliable matching
pdf_df["file_key"] = pdf_df["filename"].str.strip().str.lower()

# Create the same standardized key from the metadata dataset
gbv["file_key"] = (
    gbv["PDF_filename"]
        .str.strip()
        .str.lower()
)


In [ ]:
# Identify filenames that exist in both datasets
# This helps verify that the merge will correctly match metadata with extracted text

matches = set(pdf_df["file_key"]) & set(gbv["file_key"])
print(f"Matching files: {len(matches)}")

In [ ]:
# Merge the metadata dataset with the extracted PDF text corpus
# The merge is performed using the standardized filename key

merged_df = gbv.merge(
    pdf_df,
    on="file_key",          # shared key created earlier
    how="left",             # keep all case metadata records
    suffixes=("_details", "_pdf")  # distinguish overlapping column names
)

In [ ]:
# Quality control of merged dataset

merged_df["status"].value_counts()

In [ ]:
# Save the final dataset

merged_df.to_csv("05_gbv_analysis_dataset.csv", index=False)

## Final outputs

At the end of the notebook, the following files should be available:

- `01_case_links.csv` — list of scraped case result pages, including case titles, links and dates  
- `02_case_metadata_20.csv` — structured case metadata extracted from individual case pages, including court information, citations, dates and PDF links  
- `03_all_details_20.csv` — enriched case metadata with standardized PDF filenames  
- `04_case_text.csv` — extracted judgment text corpus  
- `05_gbv_analysis_dataset.csv` — merged dataset linking case metadata to extracted text

These outputs form the basis for subsequent GBV-focused filtering, review and classification.